In [2]:
import requests
import pandas as pd
import time
import math
from datetime import datetime
import pytz

from sql_db_func import MySQLConnetFunc
dbhandler = MySQLConnetFunc('l6a01_project')

#台灣時間轉UTC時間
def taiwan_to_utc_iso(taiwan_time_str):
    """
    將台灣時間字串轉成 UTC ISO 8601 格式 (帶 Z)
    taiwan_time_str 格式範例: "2025-11-18 16:00:00"
    """
    # 設定台灣時區
    tz_tw = pytz.timezone("Asia/Taipei")
    # 解析字串成 datetime
    dt_tw = datetime.strptime(taiwan_time_str, "%Y-%m-%d %H:%M:%S")
    # 套上台灣時區
    dt_tw = tz_tw.localize(dt_tw)
    # 轉成 UTC
    dt_utc = dt_tw.astimezone(pytz.utc)
    # 轉成 ISO 8601 字串，結尾加 Z 表示 UTC
    return dt_utc.strftime("%Y-%m-%dT%H:%M:%S.000Z")

proxyDict = {
    "http": "http://10.97.4.1:8080",
    "https": "http://10.97.4.1:8080"
}



In [3]:
# ================================
# 登入 API 取得 Token
# ================================
login_url = 'http://10.97.140.46/api/authority/login'
payload = {
    "_id": "",
    "name": "fpd",
    "password": "fpd",
    "resources": [],
    "isORM": False
}
headers = {"Content-Type": "application/json"}

response = requests.put(login_url, json=payload, proxies=proxyDict, headers=headers)
if response.status_code == 200:
    login_data = response.json()
    token = login_data['token_set']['access_token']
    print("成功登入，取得 Token")
else:
    print("登入失敗，狀態碼：", response.status_code)
    exit()

time.sleep(1)

# ================================
# 設定 API 與 headers
# ================================
BASE_URL = "http://10.97.140.46/api/search/glasses"
headers = {
    'Authorization': f'Bearer {token}',
    'Accept': 'application/json',
    'user-agent': 'Mozilla/5.0'
}

成功登入，取得 Token


In [4]:
# 只抓 CAAOI300
# 台灣時間，資料範圍時間改這裡
start_tw = "2025-08-01 00:00:00"
end_tw   = "2025-11-19 00:00:00"
# 台灣時間轉UTC時間，因KLA YMS網頁API是讀UTC時區
start_utc = taiwan_to_utc_iso(start_tw)
end_utc   = taiwan_to_utc_iso(end_tw)
page_items = 100 #每頁幾筆資料
params_common = [
    ("pastDays", 0),
    ("pastfromto", 1),
    ("machineId", "CAAOI300"), 
    ("startTime", start_utc),  # 可調日期
    ("endTime", end_utc),
    ("dataType", 1),
    ("pageItems", page_items), # 每頁資料筆數
    ("authority", "CAAOI300"), 
    ("isInit", "false")
]


In [7]:
# ================================
# 抓取資料
# ================================
all_data = []
page = 1

while True:
    print(f"抓取第 {page} 頁...")
    params_page = params_common + [("currentPage", page)]
    res = requests.get(BASE_URL, headers=headers, proxies=proxyDict, params=params_page)
    data_list = res.json().get("data", [])
    if not data_list:
        print("資料抓取完畢")
        break
    all_data.extend(data_list)
    # 如果本頁資料少於 page_items，代表已經是最後一頁
    if len(data_list) < page_items:
        print("最後一頁抓完")
        break
    page += 1
    # time.sleep(0.2)

print("全部抓取完成，總筆數:", len(all_data))
df = pd.DataFrame(all_data)

抓取第 1 頁...
抓取第 2 頁...
抓取第 3 頁...
抓取第 4 頁...
抓取第 5 頁...
抓取第 6 頁...
抓取第 7 頁...
抓取第 8 頁...
抓取第 9 頁...
抓取第 10 頁...
抓取第 11 頁...
抓取第 12 頁...
抓取第 13 頁...
抓取第 14 頁...
抓取第 15 頁...
抓取第 16 頁...
抓取第 17 頁...
抓取第 18 頁...
抓取第 19 頁...
抓取第 20 頁...
抓取第 21 頁...
抓取第 22 頁...
抓取第 23 頁...
抓取第 24 頁...
抓取第 25 頁...
抓取第 26 頁...
抓取第 27 頁...
抓取第 28 頁...
抓取第 29 頁...
抓取第 30 頁...
抓取第 31 頁...
抓取第 32 頁...
抓取第 33 頁...
抓取第 34 頁...
抓取第 35 頁...
抓取第 36 頁...
抓取第 37 頁...
抓取第 38 頁...
抓取第 39 頁...
抓取第 40 頁...
抓取第 41 頁...
抓取第 42 頁...
抓取第 43 頁...
抓取第 44 頁...
抓取第 45 頁...
抓取第 46 頁...
抓取第 47 頁...
抓取第 48 頁...
抓取第 49 頁...
抓取第 50 頁...
抓取第 51 頁...
抓取第 52 頁...
抓取第 53 頁...
抓取第 54 頁...
抓取第 55 頁...
抓取第 56 頁...
抓取第 57 頁...
抓取第 58 頁...
抓取第 59 頁...
抓取第 60 頁...
抓取第 61 頁...
抓取第 62 頁...
抓取第 63 頁...
抓取第 64 頁...
抓取第 65 頁...
抓取第 66 頁...
抓取第 67 頁...
抓取第 68 頁...
抓取第 69 頁...
抓取第 70 頁...
抓取第 71 頁...
抓取第 72 頁...
抓取第 73 頁...
抓取第 74 頁...
抓取第 75 頁...
抓取第 76 頁...
抓取第 77 頁...
抓取第 78 頁...
抓取第 79 頁...
抓取第 80 頁...
抓取第 81 頁...
抓取第 82 頁...
抓取第 83 頁...
抓取第 84 頁...
抓

In [8]:
# UTC時區轉台灣時區
df["scantime"] = (
    pd.to_datetime(df["startTime"], utc=True)   # 轉成 UTC-aware datetime
      .dt.tz_convert("Asia/Taipei")             # 轉台灣時間
      .dt.strftime("%Y-%m-%d %H:%M:%S")         # 轉成字串
)

df["run_day"] = (
    pd.to_datetime(df["startTime"], utc=True)   # 轉成 UTC-aware datetime
      .dt.tz_convert("Asia/Taipei")             # 轉台灣時間
      .dt.strftime("%Y-%m-%d")         # 轉成字串
)

df["endTime_tw"] = (
    pd.to_datetime(df["endTime"], utc=True)   # 轉成 UTC-aware datetime
      .dt.tz_convert("Asia/Taipei")             # 轉台灣時間
      .dt.strftime("%Y-%m-%d %H:%M:%S")         # 轉成字串
)


# 欄位改名，可有可無
df_renamed = df.rename(columns={
    'glassId': 'glass_id',
    'layer': 'model_id',
    'device': 'recipe_id',
    'lotId': 'lot_id',
    'machineId': 'aoi_id',
})
# 撈特定欄位資料
columns = [ 'run_day', 'scantime', 'glass_id', 'recipe_id' ] 
df_renamed = df_renamed[columns]
print(df_renamed.head())
# df.to_csv("glass_data_CAAOI300.csv", index=False)


      run_day             scantime   glass_id        recipe_id
0  2025-11-18  2025-11-18 23:58:53  JF5ABJA0K  B140UAN04-T-BPI
1  2025-11-18  2025-11-18 23:53:03  JF5ABJA0J  B140UAN04-T-BPI
2  2025-11-18  2025-11-18 23:47:04  JF5ABJA0H  B140UAN04-T-BPI
3  2025-11-18  2025-11-18 23:41:16  JF5ABJA0G  B140UAN04-T-BPI
4  2025-11-18  2025-11-18 23:35:30  JF5ABJA0F  B140UAN04-T-BPI


In [18]:
tbn = 'aoi_summary_aoi300_capa'
#dbhandler.save_df( df_renamed, tbn)
df = dbhandler.get_table(tbn)
len(df)
df


,run_day,scantime,glass_id,recipe_id
0,2025-08-01,2025-08-01 00:02:59,YD5A7NN8M,B160UAN4A-T-BPI
1,2025-08-01,2025-08-01 00:08:47,YD5A7NN8N,B160UAN4A-T-BPI
2,2025-08-01,2025-08-01 00:14:35,YD5A7NN8P,B160UAN4A-T-BPI
3,2025-08-01,2025-08-01 00:20:34,YD5A7NN8Q,B160UAN4A-T-BPI
4,2025-08-01,2025-08-01 00:26:23,YD5A7NN8R,B160UAN4A-T-BPI
...,...,...,...,...
20259,2025-11-20,2025-11-20 15:23:25,MQ5ABPX7B,CELL-ITO
20260,2025-11-20,2025-11-20 15:29:04,MQ5ABRU2K,CELL-ITO
20261,2025-11-20,2025-11-20 15:34:43,MQ5ABRU2J,CELL-ITO
20262,2025-11-20,2025-11-20 16:00:47,MQ5ABRU4K,CELL-ITO
